# Certified Bessel-kernel minimax search

Self-contained Colab notebook with the source and saved candidates for dimensions 52,56,...,100. Run all cells in order.

The default dimension schedule is 2,3,...,20; 22,24,...,36; 40,44,...,100. Change DIMENSIONS or D_MAX in the settings cell. All integer d>=2, including odd dimensions, are supported.

MODE="auto" starts with a saved candidate where available, then searches further if it misses the target; dimensions without saved candidates start with optimization. MODE="optimize" always starts a new search. MODE="verify_saved" only checks embedded candidates. No candidate is claimed to be a global optimizer.

Targets are set by TARGET_ANCHORS, with linear interpolation between anchors and a constant value outside them. Defaults are [(2,.765),(4,.765),(6,.770),(52,.789)]; thus d=3 uses .765 and d=5 uses .7675. TARGET_OVERRIDES can set individual dimensions. The applicable target is printed and saved with every result. Not meeting a target is reported as target_achieved=False; the valid bound is still reported. A target does not guarantee convergence within the chosen search and certification budgets.

No plots are generated. Every dimension prints a clearly labelled certified upper bound on I^(1/d), a certified lower bound on (4 |B_1| I)^(-1/d), and that lower bound divided by sqrt(d). The derived bounds are rounded downward to six decimals. Large coefficients must remain decimal strings to preserve their certificates. Plots are rendered directly; plotting failures do not discard certificates.

The kernel uses Gamma(d/2) J_(d/2-1)(2*pi*r)/(pi*r)^(d/2-1), continuously extended by 1 at zero. Arb ball arithmetic, adaptive Taylor bounds, and an infinite-tail envelope certify each candidate over all r>0. For odd dimensions the formulas use the exact half-integer d/2. For d=2 the decaying tail bound is |J_0(x)|<=3/sqrt(x). If a radius or evaluation limit is reached, the remaining bounds stay in the result, which may be loose.

The d=2 tail bound follows by splitting the integral representation of J_0 over [0,pi] at distance delta=1/sqrt(x) from both endpoints (x>=1). Endpoints contribute at most 2delta; integration by parts on each monotone half of the middle interval gives total at most 4/[x sin(delta)]. Using sin(delta)>=2delta/pi gives (2+2/pi)/sqrt(x)<3/sqrt(x) after dividing by pi. For x<1, use |J_0(x)|<=1. Other dimensions use |J_nu(x)|<=1 for nu>=0: https://dlmf.nist.gov/10.14.E1.

Solver retries and a best-previous-proposal checkpoint handle some numerical LP failures. ALLOW_SAVED_FALLBACK explicitly permits a saved candidate when a new optimization fails. Such results are labelled as saved fallbacks.

Certification now prints progress every CERT_PROGRESS_SECONDS, uses a soft CERT_TIME_LIMIT, and can stop when the certified interval for I^(1/d) is narrower than CERT_ROOT_GAP. It warns when a rigorously evaluated point exceeds the target for the current candidate. A budget-limited result is still a valid upper bound, but may be loose; the interval and stop_reason are saved. Limits are checked between evaluations, so a single expensive evaluation may overrun the time limit.

The exchange loop adds high-error points discovered by certification to the next optimization grid. All added points are retained. EXCHANGE_ROUNDS limits the number of cycles, and EXCHANGE_POINTS limits the new points per cycle. Compatible previous coefficients initialize the next solve. The final result is the smallest certified error bound among the completed rounds; later solver failures or weaker certificates cannot overwrite it. Individual rounds and the best certified checkpoint are saved. A tighter certificate is not necessarily a smaller true candidate error, so histories retain both trial bounds and the best bound. Verification-only mode performs one certificate and does not optimize.

## CSV export

The run writes `results.csv` after every completed dimension and automatically downloads it at the end in Colab. Set `AUTO_DOWNLOAD_CSV=False` to disable the automatic download. No plots are generated.

Columns: `d`, `I_root_upper_bound` (the certified upper bound on I^(1/d)), `derived_lower_bound` (the lower bound on (4|B_1|I)^(-1/d)), `derived_over_sqrt_d_lower_bound`, `target_root`, `target_achieved`, and `candidate_origin`. The three reported bounds retain the same conservative six-decimal rounding as the printed output. Failed dimensions have no certified row.

If you interrupt the run, the CSV still contains the dimensions already completed. Run `download_results_csv()` in a new cell, or download `results.csv` from the Colab Files panel under `/content/kappa_certified_run`.


In [ ]:
%pip install -q python-flint scipy numpy matplotlib pillow


In [ ]:
from pathlib import Path
import sys, json, math, importlib, subprocess
ROOT = Path("/content/kappa_certified_run") if Path("/content").exists() else Path.cwd()/"kappa_certified_run"
ROOT.mkdir(parents=True, exist_ok=True)

ALLOW_SAVED_FALLBACK = True  # explicitly use a saved candidate if optimization fails
MODE = "auto"  # auto: verify saved candidates, optimize dimensions without one
D_MAX = 100
DIMENSIONS = (list(range(2, min(D_MAX, 20)+1))
              + list(range(22, min(D_MAX, 36)+1, 2))
              + list(range(40, D_MAX+1, 4)))
N_FREQ = 48
REFINEMENT_STEPS = 5
EXCHANGE_ROUNDS = 5       # optimization/certification cycles per dimension
EXCHANGE_POINTS = 24      # new peak/interval points added per cycle
A_MIN, A_MAX = 1.002, 2.0  # a_j = exp(p_j)
# Piecewise-linear targets; constant beyond the first/last anchor.
TARGET_ANCHORS = [(2, 0.765), (4, 0.765), (6, 0.770), (52, 0.789)]
TARGET_OVERRIDES = {}  # optional, for example {3: 0.766}

def target_root_for_dimension(d):
    if d in TARGET_OVERRIDES:
        return float(TARGET_OVERRIDES[d])
    anchors = sorted(TARGET_ANCHORS)
    if not anchors or any(x[1] <= 0 for x in anchors):
        raise ValueError("Target anchors must be nonempty with positive values")
    if any(anchors[i][0] == anchors[i+1][0] for i in range(len(anchors)-1)):
        raise ValueError("Target anchor dimensions must be distinct")
    if d <= anchors[0][0]:
        return float(anchors[0][1])
    for (left, low), (right, high) in zip(anchors, anchors[1:]):
        if d <= right:
            return float(low + (high-low)*(d-left)/(right-left))
    return float(anchors[-1][1])
FIT_DIGITS = 100
CERT_DIGITS = 200
MAX_CERT_EVALS = 10000
CERT_TIME_LIMIT = 120     # seconds, checked between interval evaluations
CERT_ROOT_GAP = 5e-5      # stop when the certified root interval is this narrow
CERT_PROGRESS_SECONDS = 10
MAX_CERT_RADIUS = 1048576  # finite radius cap; any remaining tail stays in the bound
CERT_TOL = 1e-15

AUTO_DOWNLOAD_CSV = True  # download results.csv when the run finishes in Colab


In [ ]:
# Embedded source: kernel.py
source = r'''
"""Certified Bessel minimax candidates for integer d>=2. Colab: !pip install python-flint scipy
Run: !python kappa_certified.py --start 4 --stop 52
Each printed I is an Arb-certified upper bound for the chosen exact decimal
coefficients and frequencies; it is not a proof of global optimality.
The derived number is rounded down from (4 |B_1| I_upper)^(-1/d).
The code finds coefficients by discretized LP and certifies using Arb, an
adaptive second-derivative bound, and a Bessel tail bound.
"""
import argparse,heapq,json,math,time
import numpy as np
from scipy.special import jv,gammaln
from scipy.optimize import linprog
from flint import arb,ctx
ctx.dps=200

def kap(x,d):
 x=np.asarray(x,dtype=float);v=d/2-1;o=np.ones_like(x);m=x>1e-9
 o[m]=np.exp(gammaln(d/2)-v*np.log(np.pi*x[m]))*jv(v,2*np.pi*x[m]);return o

def optimize(d,n,amin,amax,cmax):
 a=np.geomspace(amin,amax,n);t=np.r_[np.linspace(0,20,8001),np.geomspace(20.001,100,1200)]
 A=kap(t[:,None]*a,d)-a**(-d)*kap(t[:,None]/a,d);b=kap(t,d);m=len(t)
 r=linprog(np.r_[np.zeros(n),1.],A_ub=np.r_[np.c_[-A,-np.ones(m)],np.c_[A,-np.ones(m)]],b_ub=np.r_[-b,b],bounds=[(-cmax,cmax)]*n+[(0,None)],method='highs',options={'small_matrix_value':1e-12,'primal_feasibility_tolerance':1e-9,'dual_feasibility_tolerance':1e-9})
 if r.x is None:raise RuntimeError('LP failed: '+r.message)
 return a,r.x[:-1],r.fun

def up(x):return math.nextafter(float(x.upper()),math.inf)
def down(x):return math.nextafter(float(x.lower()),-math.inf)
def kball(t,d):
 if t.is_zero():return arb(1)
 alpha=arb(d)/2;nu=alpha-1;pi=arb.pi();return alpha.gamma()*(2*pi*t).bessel_j(nu)/(pi*t)**nu

def certify(d,af,cf,tol,maxeval,target_root=None,max_radius=1048576,time_limit=120,root_tol=5e-5,progress_seconds=10):
 if not isinstance(d,int) or d<2:raise ValueError("Requires integer d >= 2")
 if max_radius<10:raise ValueError("max_radius must be at least 10")
 a=[arb(x if isinstance(x,str) else repr(float(x))) for x in af];c=[arb(x if isinstance(x,str) else repr(float(x))) for x in cf];alpha=arb(d)/2;nu=alpha-1;pi=arb.pi()
 # |J_nu(x)| <= 1 for real nu >= 0 (DLMF 10.14.1).
 # For d=2, use the decaying bound |J_0(x)| <= 3/sqrt(x).
 # Its elementary proof is included in the notebook notes.
 beta=arb(1)/2 if d==2 else nu
 prefactor=arb(3)/(2*pi).sqrt() if d==2 else alpha.gamma()/pi**nu
 K=prefactor*(1+sum(abs(ci)*(ai**(-beta)+ai**(-d+beta)) for ai,ci in zip(a,c)))
 global_bound=up(1+sum(abs(ci)*(1+ai**(-d)) for ai,ci in zip(a,c)))
 # From the spherical integral, E|omega_1|^3 <= E omega_1^2 = 1/d.
 moment16=arb(math.prod(range(1,16,2)))/math.prod(range(d,d+16,2))
 M9=(2*pi)**17*moment16*(1+sum(abs(ci)*(ai**17+ai**(-d-17)) for ai,ci in zip(a,c)))
 orders=(0,)+tuple(range(2,17))
 terms={}
 for n in orders[1:]:
  terms[n]=[]
  for j in range(n//2+1):
   m=n-j;power=n-2*j;rising=math.prod(alpha+k for k in range(m))
   factor=arb(math.factorial(n)*2**power)/(math.factorial(j)*math.factorial(power)*rising)*(-pi*pi)**m
   terms[n].append((m,power,factor))
 def kderivs(t):
  K=[kball(t,d),kball(t,d+2)]
  for m in range(1,16):
   K.append(((alpha+m)*(alpha+m-1))*(K[-1]-K[-2])/(arb.pi()*t)**2 if not t.is_zero() else arb(1))
  out=[K[0]]
  for n in orders[1:]:out.append(sum(factor*t**power*K[m] for m,power,factor in terms[n]))
  return out
 weights=[(tuple(ci*ai**n for n in orders),tuple(ci*ai**(-d-n) for n in orders)) for ai,ci in zip(a,c)]
 sampled_points=[]
 def point(x):
  t=arb(x);v=kball(t,d)
  for ai,ci in zip(a,c):v-=ci*(kball(ai*t,d)-ai**(-d)*kball(t/ai,d))
  hi=min(global_bound,up(abs(v)));lo=max(0.0,down(abs(v)))
  sampled_points.append((float(x),lo,hi))
  return hi,lo
 def fall(x):
  t=arb(x);vals=list(kderivs(t))
  for ai,(forward,reverse) in zip(a,weights):
   pos=kderivs(ai*t);neg=kderivs(t/ai)
   for j in range(len(orders)):vals[j]-=forward[j]*pos[j]-reverse[j]*neg[j]
  return tuple(up(abs(v)) for v in vals),max(0.0,down(abs(vals[0])))
 start=time.monotonic();last_report=start
 fx,lx=point(0);ref=max(fx,.001);R=10
 target_value=arb(repr(target_root))**d if target_root is not None else None
 target_lo=down(target_value) if target_value is not None else -1
 target_hi=up(target_value) if target_value is not None else -1
 tail_goal=min(ref*.9,target_lo*.5) if target_root is not None else ref*.9
 while up(K/arb(R)**beta)>tail_goal and R<max_radius:R=min(2*R,max_radius)
 tail=up(K/arb(R)**beta)
 fy,ly=point(R);count=2;best_lower=max(lx,ly);best_point_upper=max(fx,fy)
 # The initial global bound covers the whole finite interval immediately.
 # Every split inherits its parent's valid bound, even if a budget expires.
 heap=[(-global_bound,0.0,float(R),fx,fy)]
 warned=False;reason='evaluation_limit'
 def state():
  U=min(global_bound,max(-heap[0][0] if heap else 0.0,tail))
  ru=up(arb(U)**(arb(1)/d))
  rl=max(0.0,down(arb(best_lower)**(arb(1)/d)))
  return U,rl,ru
 print(f'CERT d={d}: starting; R={R:g}, tail_bound={tail:.6g}, time_limit={time_limit}s',flush=True)
 while heap:
  U,rl,ru=state()
  if target_root is not None and U<=target_lo:reason='target_met';break
  if up(arb(ru)-arb(rl))<=root_tol:reason='root_gap_met';break
  if U-best_lower<=tol:reason='absolute_gap_met';break
  elapsed=time.monotonic()-start
  if time_limit is not None and elapsed>=time_limit:reason='time_limit';break
  if count>=maxeval:reason='evaluation_limit';break
  if target_root is not None and best_lower>target_hi and not warned:
   print(f'CERT d={d}: a certified point already exceeds the target; '
         'this candidate cannot meet it. Continuing to bound its actual error.',flush=True)
   warned=True
  negub,x,y,fx,fy=heapq.heappop(heap);parent_ub=-negub;mid=(x+y)/2
  if mid==x or mid==y:
   heapq.heappush(heap,(negub,x,y,fx,fy));reason='floating_point_resolution';break
  values,lmid=fall(mid);fm=min(global_bound,values[0]);count+=1
  sampled_points.append((float(mid),lmid,fm))
  best_lower=max(best_lower,lmid);best_point_upper=max(best_point_upper,fm)
  h=arb(y)-arb(x);rho=arb(max(up(abs(arb(mid)-arb(x))),up(abs(arb(y)-arb(mid)))))
  curv=sum(arb(values[n-1])*rho**(n-2)/math.factorial(n-2) for n in range(2,17))+M9*rho**15/math.factorial(15)
  local_ub=min(parent_ub,global_bound,up(arb(max(fx,fy))+curv*h*h/8))
  for left,right,fl,fr in [(x,mid,fx,fm),(mid,y,fm,fy)]:
   child_ub=local_ub
   if left>0:child_ub=min(child_ub,up(K/arb(left)**beta))
   heapq.heappush(heap,(-child_ub,left,right,fl,fr))
  now=time.monotonic()
  if now-last_report>=progress_seconds:
   U,rl,ru=state()
   print(f'CERT d={d}: {count} evaluations, {now-start:.1f}s; '
         f'root in [{rl:.6g}, {ru:.6g}]',flush=True);last_report=now
 U,rl,ru=state()
 elapsed=time.monotonic()-start
 print(f'CERT d={d}: stopped ({reason}); root in [{rl:.6g}, {ru:.6g}]; '
       f'{count} evaluations, {elapsed:.1f}s',flush=True)
 # Export measured peaks plus unresolved interval centers for the next LP.
 exchange_points=[]
 separation=0.02/max(float(x.upper()) for x in a) if a else 0.02
 for x,lo,hi in sorted(sampled_points,key=lambda row:row[1],reverse=True):
  if all(abs(x-y)>separation for y in exchange_points):exchange_points.append(x)
  if len(exchange_points)>=48:break
 peak_points=list(exchange_points)
 unresolved_points=[]
 for item in sorted(heap)[:32]:
  x=(item[1]+item[2])/2
  if x not in unresolved_points:unresolved_points.append(x)
  if x not in exchange_points:exchange_points.append(x)
 peak=max(sampled_points,key=lambda row:row[1])
 return U,{'exchange_points':exchange_points,'peak_points':peak_points,
           'unresolved_points':unresolved_points,'peak_location':peak[0],
           'peak_lower':peak[1],'R':R,'tail_bound':tail,'point_upper':best_point_upper,
           'point_lower':best_lower,'evals':count,'gap':U-best_lower,
           'root_lower':rl,'root_upper':ru,'stop_reason':reason,
           'elapsed_seconds':elapsed,'tail_target_met':tail<=tail_goal,
           'max_radius':max_radius}

'''
(ROOT/'kernel.py').write_text(source, encoding="utf-8")


In [ ]:
# Embedded source: kernel17.py
source = r'''
"""Certified Bessel minimax candidates for integer d>=2. Colab: !pip install python-flint scipy
Run: !python kappa_certified.py --start 4 --stop 52
Each printed I is an Arb-certified upper bound for the chosen exact decimal
coefficients and frequencies; it is not a proof of global optimality.
The derived number is rounded down from (4 |B_1| I_upper)^(-1/d).
The code finds coefficients by discretized LP and certifies using Arb, an
adaptive second-derivative bound, and a Bessel tail bound.
"""
import argparse,heapq,json,math,time
import numpy as np
from scipy.special import jv,gammaln
from scipy.optimize import linprog
from flint import arb,ctx
ctx.dps=200

def kap(x,d):
 x=np.asarray(x,dtype=float);v=d/2-1;o=np.ones_like(x);m=x>1e-9
 o[m]=np.exp(gammaln(d/2)-v*np.log(np.pi*x[m]))*jv(v,2*np.pi*x[m]);return o

def optimize(d,n,amin,amax,cmax):
 a=np.geomspace(amin,amax,n);t=np.r_[np.linspace(0,20,8001),np.geomspace(20.001,100,1200)]
 A=kap(t[:,None]*a,d)-a**(-d)*kap(t[:,None]/a,d);b=kap(t,d);m=len(t)
 r=linprog(np.r_[np.zeros(n),1.],A_ub=np.r_[np.c_[-A,-np.ones(m)],np.c_[A,-np.ones(m)]],b_ub=np.r_[-b,b],bounds=[(-cmax,cmax)]*n+[(0,None)],method='highs',options={'small_matrix_value':1e-12,'primal_feasibility_tolerance':1e-9,'dual_feasibility_tolerance':1e-9})
 if r.x is None:raise RuntimeError('LP failed: '+r.message)
 return a,r.x[:-1],r.fun

def up(x):return math.nextafter(float(x.upper()),math.inf)
def down(x):return math.nextafter(float(x.lower()),-math.inf)
def kball(t,d):
 if t.is_zero():return arb(1)
 alpha=arb(d)/2;nu=alpha-1;pi=arb.pi();return alpha.gamma()*(2*pi*t).bessel_j(nu)/(pi*t)**nu

def certify(d,af,cf,tol,maxeval,target_root=None,max_radius=1048576,time_limit=120,root_tol=5e-5,progress_seconds=10):
 if not isinstance(d,int) or d<2:raise ValueError("Requires integer d >= 2")
 if max_radius<10:raise ValueError("max_radius must be at least 10")
 a=[arb(x if isinstance(x,str) else repr(float(x))) for x in af];c=[arb(x if isinstance(x,str) else repr(float(x))) for x in cf];alpha=arb(d)/2;nu=alpha-1;pi=arb.pi()
 # |J_nu(x)| <= 1 for real nu >= 0 (DLMF 10.14.1).
 # For d=2, use the decaying bound |J_0(x)| <= 3/sqrt(x).
 # Its elementary proof is included in the notebook notes.
 beta=arb(1)/2 if d==2 else nu
 prefactor=arb(3)/(2*pi).sqrt() if d==2 else alpha.gamma()/pi**nu
 K=prefactor*(1+sum(abs(ci)*(ai**(-beta)+ai**(-d+beta)) for ai,ci in zip(a,c)))
 global_bound=up(1+sum(abs(ci)*(1+ai**(-d)) for ai,ci in zip(a,c)))
 # From the spherical integral, E|omega_1|^3 <= E omega_1^2 = 1/d.
 moment16=arb(math.prod(range(1,16,2)))/math.prod(range(d,d+16,2))
 M9=(2*pi)**17*moment16*(1+sum(abs(ci)*(ai**17+ai**(-d-17)) for ai,ci in zip(a,c)))
 orders=(0,)+tuple(range(2,17))
 terms={}
 for n in orders[1:]:
  terms[n]=[]
  for j in range(n//2+1):
   m=n-j;power=n-2*j;rising=math.prod(alpha+k for k in range(m))
   factor=arb(math.factorial(n)*2**power)/(math.factorial(j)*math.factorial(power)*rising)*(-pi*pi)**m
   terms[n].append((m,power,factor))
 def kderivs(t):
  K=[kball(t,d),kball(t,d+2)]
  for m in range(1,16):
   K.append(((alpha+m)*(alpha+m-1))*(K[-1]-K[-2])/(arb.pi()*t)**2 if not t.is_zero() else arb(1))
  out=[K[0]]
  for n in orders[1:]:out.append(sum(factor*t**power*K[m] for m,power,factor in terms[n]))
  return out
 weights=[(tuple(ci*ai**n for n in orders),tuple(ci*ai**(-d-n) for n in orders)) for ai,ci in zip(a,c)]
 sampled_points=[]
 def point(x):
  t=arb(x);v=kball(t,d)
  for ai,ci in zip(a,c):v-=ci*(kball(ai*t,d)-ai**(-d)*kball(t/ai,d))
  hi=min(global_bound,up(abs(v)));lo=max(0.0,down(abs(v)))
  sampled_points.append((float(x),lo,hi))
  return hi,lo
 def fall(x):
  t=arb(x);vals=list(kderivs(t))
  for ai,(forward,reverse) in zip(a,weights):
   pos=kderivs(ai*t);neg=kderivs(t/ai)
   for j in range(len(orders)):vals[j]-=forward[j]*pos[j]-reverse[j]*neg[j]
  return tuple(up(abs(v)) for v in vals),max(0.0,down(abs(vals[0])))
 start=time.monotonic();last_report=start
 fx,lx=point(0);ref=max(fx,.001);R=10
 target_value=arb(repr(target_root))**d if target_root is not None else None
 target_lo=down(target_value) if target_value is not None else -1
 target_hi=up(target_value) if target_value is not None else -1
 tail_goal=min(ref*.9,target_lo*.5) if target_root is not None else ref*.9
 while up(K/arb(R)**beta)>tail_goal and R<max_radius:R=min(2*R,max_radius)
 tail=up(K/arb(R)**beta)
 fy,ly=point(R);count=2;best_lower=max(lx,ly);best_point_upper=max(fx,fy)
 # The initial global bound covers the whole finite interval immediately.
 # Every split inherits its parent's valid bound, even if a budget expires.
 heap=[(-global_bound,0.0,float(R),fx,fy)]
 warned=False;reason='evaluation_limit'
 def state():
  U=min(global_bound,max(-heap[0][0] if heap else 0.0,tail))
  ru=up(arb(U)**(arb(1)/d))
  rl=max(0.0,down(arb(best_lower)**(arb(1)/d)))
  return U,rl,ru
 print(f'CERT d={d}: starting; R={R:g}, tail_bound={tail:.6g}, time_limit={time_limit}s',flush=True)
 while heap:
  U,rl,ru=state()
  if target_root is not None and U<=target_lo:reason='target_met';break
  if up(arb(ru)-arb(rl))<=root_tol:reason='root_gap_met';break
  if U-best_lower<=tol:reason='absolute_gap_met';break
  elapsed=time.monotonic()-start
  if time_limit is not None and elapsed>=time_limit:reason='time_limit';break
  if count>=maxeval:reason='evaluation_limit';break
  if target_root is not None and best_lower>target_hi and not warned:
   print(f'CERT d={d}: a certified point already exceeds the target; '
         'this candidate cannot meet it. Continuing to bound its actual error.',flush=True)
   warned=True
  negub,x,y,fx,fy=heapq.heappop(heap);parent_ub=-negub;mid=(x+y)/2
  if mid==x or mid==y:
   heapq.heappush(heap,(negub,x,y,fx,fy));reason='floating_point_resolution';break
  values,lmid=fall(mid);fm=min(global_bound,values[0]);count+=1
  sampled_points.append((float(mid),lmid,fm))
  best_lower=max(best_lower,lmid);best_point_upper=max(best_point_upper,fm)
  h=arb(y)-arb(x);rho=arb(max(up(abs(arb(mid)-arb(x))),up(abs(arb(y)-arb(mid)))))
  curv=sum(arb(values[n-1])*rho**(n-2)/math.factorial(n-2) for n in range(2,17))+M9*rho**15/math.factorial(15)
  local_ub=min(parent_ub,global_bound,up(arb(max(fx,fy))+curv*h*h/8))
  for left,right,fl,fr in [(x,mid,fx,fm),(mid,y,fm,fy)]:
   child_ub=local_ub
   if left>0:child_ub=min(child_ub,up(K/arb(left)**beta))
   heapq.heappush(heap,(-child_ub,left,right,fl,fr))
  now=time.monotonic()
  if now-last_report>=progress_seconds:
   U,rl,ru=state()
   print(f'CERT d={d}: {count} evaluations, {now-start:.1f}s; '
         f'root in [{rl:.6g}, {ru:.6g}]',flush=True);last_report=now
 U,rl,ru=state()
 elapsed=time.monotonic()-start
 print(f'CERT d={d}: stopped ({reason}); root in [{rl:.6g}, {ru:.6g}]; '
       f'{count} evaluations, {elapsed:.1f}s',flush=True)
 # Export measured peaks plus unresolved interval centers for the next LP.
 exchange_points=[]
 separation=0.02/max(float(x.upper()) for x in a) if a else 0.02
 for x,lo,hi in sorted(sampled_points,key=lambda row:row[1],reverse=True):
  if all(abs(x-y)>separation for y in exchange_points):exchange_points.append(x)
  if len(exchange_points)>=48:break
 peak_points=list(exchange_points)
 unresolved_points=[]
 for item in sorted(heap)[:32]:
  x=(item[1]+item[2])/2
  if x not in unresolved_points:unresolved_points.append(x)
  if x not in exchange_points:exchange_points.append(x)
 peak=max(sampled_points,key=lambda row:row[1])
 return U,{'exchange_points':exchange_points,'peak_points':peak_points,
           'unresolved_points':unresolved_points,'peak_location':peak[0],
           'peak_lower':peak[1],'R':R,'tail_bound':tail,'point_upper':best_point_upper,
           'point_lower':best_lower,'evals':count,'gap':U-best_lower,
           'root_lower':rl,'root_upper':ru,'stop_reason':reason,
           'elapsed_seconds':elapsed,'tail_target_met':tail<=tail_goal,
           'max_radius':max_radius}

'''
(ROOT/'kernel17.py').write_text(source, encoding="utf-8")


In [ ]:
# Embedded source: precond.py
source = r'''
import sys,time,json,math
from pathlib import Path
from fractions import Fraction
import numpy as np
from scipy.linalg import svd
from scipy.optimize import linprog
from flint import arb,arb_mat,ctx
import kernel as k
ctx.dps=100
root=Path(__file__).resolve().parent

def solve_refinement(BF, rhs):
    """Find an LP proposal with retries; certification happens separately."""
    m, n = BF.shape
    if not np.all(np.isfinite(BF)) or not np.all(np.isfinite(rhs)):
        raise RuntimeError("Nonfinite proposal matrix; increase FIT_DIGITS")
    colscale = np.max(np.abs(BF), axis=0)
    colscale[colscale == 0] = 1.0
    attempts = [("highs", False, None, True),
                ("highs-ds", True, 1e6, True),
                ("highs-ipm", True, 1e6, True),
                ("highs-ds", True, 1e4, False)]
    errors = []
    for method, scaled, limit, presolve in attempts:
        factors = colscale if scaled else np.ones(n)
        matrix = BF / factors
        bounds = [(-limit, limit) if limit is not None else (None, None)]*n + [(0, None)]
        try:
            result = linprog(np.r_[np.zeros(n), 1.],
                A_ub=np.r_[np.c_[matrix, -np.ones(m)], np.c_[-matrix, -np.ones(m)]],
                b_ub=np.r_[rhs, -rhs], bounds=bounds, method=method,
                options={"presolve": presolve, "time_limit": 30,
                         "primal_feasibility_tolerance": 1e-9,
                         "dual_feasibility_tolerance": 1e-9})
            if result.success and result.x is not None and np.all(np.isfinite(result.x)):
                return result.x[:n] / factors
            errors.append(str(result.message))
        except Exception as exc:
            errors.append(str(exc))
        print(f"PRECOND solver retry: {method} did not return a usable proposal", flush=True)
    raise RuntimeError("All LP retries failed: " + " | ".join(errors))

def fit(d,n=48,steps=5,amin=1.002,amax=2.,extra_points=None,initial_candidate=None):
 a=np.geomspace(amin,amax,n)
 base_grid=np.r_[np.linspace(0,15,701),np.geomspace(15.01,60,100)]
 extra_points=sorted(set(float(x) for x in (extra_points or []) if math.isfinite(float(x)) and float(x)>=0))
 grid=np.unique(np.r_[base_grid,extra_points])
 print(f'PRECOND d={d}: grid has {len(grid)} points; {len(extra_points)} exchange points; max r={max(grid):g}',flush=True)
 aa=[arb(repr(float(x))) for x in a];AR=[];bb=[]
 for t in grid:
  z=arb(repr(float(t)));bb.append(k.kball(z,d));AR.append([k.kball(x*z,d)-x**(-d)*k.kball(z/x,d) for x in aa])
 A=arb_mat(AR);b=arb_mat([[v] for v in bb]);AF=np.array([[float(x.mid()) for x in row] for row in AR]);_,s,V=svd(AF,full_matrices=False)
 DD=V.T/np.maximum(s,1e-16)[None,:]
 D=arb_mat([[arb(repr(float(x))) for x in row] for row in DD]);B=A*D;BF=np.array([[float(x.mid()) for x in row] for row in B.tolist()]);m=len(grid)
 C=arb_mat(n,1);best=None
 if initial_candidate is not None and len(initial_candidate['a'])==n and all(
     Fraction(str(x))==Fraction(repr(float(y))) for x,y in zip(initial_candidate['a'],a)):
  C=arb_mat([[arb(x)] for x in initial_candidate['c']])
  print(f'PRECOND d={d}: starting from the previous coefficients',flush=True)
 E=b-A*C
 if initial_candidate is not None and any(not C[i,0].is_zero() for i in range(n)):
  residual=max(float(abs(E[i,0]).upper()) for i in range(m))
  best={'d':d,'a':[repr(float(x)) for x in a],
        'c':[C[i,0].mid().str(95,radius=False) for i in range(n)],
        'sampled_error':residual,'fit_grid_size':m,'extra_grid_points':extra_points}

 for step in range(steps):
  ef=np.array([float(E[i,0].mid()) for i in range(m)]);scale=max(abs(ef));rhs=ef/scale
  try:
   proposal=solve_refinement(BF,rhs)
  except RuntimeError as exc:
   if best is None:raise
   print(f"PRECOND refinement stopped: {exc}. Retaining the best earlier candidate.",flush=True)
   break
  dz=arb_mat([[arb(repr(float(v)))*arb(repr(float(scale)))] for v in proposal])
  C=C+D*dz;E=b-A*C
  residual=max(float(abs(E[i,0]).upper()) for i in range(m))
  # Save decimal midpoints as the exact candidate coefficients, not balls.
  cs=[C[i,0].mid().str(95,radius=False) for i in range(n)]
  row={'d':d,'a':[repr(float(x)) for x in a],'c':cs,'sampled_error':residual,'fit_grid_size':m,'extra_grid_points':extra_points}
  if best is None or residual<best['sampled_error']:best=row
  (root/f'precond_d{d}.json').write_text(json.dumps(best))
  print(f'PRECOND d={d} step={step} sampled_root={residual**(1/d):.9f} max_c={max(abs(float(C[i,0].mid())) for i in range(n)):.3g}',flush=True)
 return best
if __name__=='__main__':fit(int(sys.argv[1]))

'''
(ROOT/'precond.py').write_text(source, encoding="utf-8")


In [ ]:
# Exact decimal strings of the saved candidates
SAVED_CANDIDATES = json.loads(r'''
{"52": {"d": 52, "upper_unrounded": 4.658102793639798e-06, "root": 0.7897062205875915, "derived": 2.259581173698042, "a": ["1.005", "1.0127258445937721", "1.020511080903651", "1.0283561654979638", "1.036261558454864", "1.0442277233893145", "1.052255127480274", "1.0603442414980977", "1.0684955398321432", "1.076709500518593", "1.0849866052684882", "1.0933273394959797", "1.1017321923467942", "1.1102016567269208", "1.1187362293315186", "1.1273364106740442", "1.1360027051156052", "1.1447356208945385", "1.1535356701562167", "1.1624033689830824", "1.1713392374249139", "1.1803437995293244", "1.189417583372495", "1.1985611210901428", "1.20777494890873", "1.217059607176909", "1.226415640397214", "1.2358435972579906", "1.2453440306655765", "1.254917497776725", "1.264564560031281", "1.2742857831851055", "1.2840817373432563", "1.2939529969934207", "1.3039001410396067", "1.313923752836094", "1.3240244202216442", "1.3342027355539745", "1.3444592957444976", "1.3547947022933282", "1.3652095613245563", "1.3757044836217958", "1.3862800846640024", "1.3969369846615702", "1.4076758085927028", "1.418497186240066", "1.4294017522277216", "1.4403901460583457", "1.4514630121507317", "1.4626209998775825", "1.4738647636035946", "1.4851949627238317", "1.4966122617023967", "1.5081173301113986", "1.5197108426702197", "1.5313934792850863", "1.54316592508894", "1.5550288704816186", "1.5669830111703453", "1.5790290482105278", "1.5911676880468728", "1.6033996425548152", "1.6157256290822666", "1.628146370491684", "1.6406625952024634", "1.6532750372336569", "1.6659844362470204", "1.6787915375903915", "1.6916970923413996", "1.7047018573515154", "1.7178065952904336", "1.7310120746908026", "1.7443190699932947", "1.7577283615920227", "1.7712407358803073", "1.7848569852967948", "1.7985779083719313", "1.8124043097747906", "1.8263370003602666", "1.8403767972166245", "1.8545245237134205", "1.868781009549788", "1.8831470908030963", "1.8976236099779809", "1.9122114160557553", "1.9269113645441969", "1.9417243175277215", "1.9566511437179384", "1.9716927185045967", "1.9868499240069242", "2.0021236491253576", "2.0175147895936747", "2.0330242480315235", "2.048652933997357", "2.064401764041774", "2.0802716617612727", "2.096263557852413", "2.1123783901663993", "2.1286171037640793", "2.1449806509713696", "2.1614699914351045", "2.1780860921793135", "2.1948299276619347", "2.2117024798319607", "2.228704738187028", "2.245837699831443", "2.2631023695346597", "2.280499759790205", "2.2980308908750544", "2.3156967909094712", "2.333498495917296", "2.3514370498867074", "2.3695135048314473", "2.387728920852514", "2.406084366200337", "2.424580917337421", "2.4432196590014756", "2.462001684269032", "2.4809280946195478", "2.5"], "c": ["150000.0", "150000.0", "150000.0", "98634.42149445077", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "150000.0", "150000.0", "132746.94534375903", "150000.0", "150000.0", "144424.61218299222", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "22843.360693450384", "150000.0", "150000.0", "150000.0", "150000.0", "150000.0", "150000.0", "-86272.9630193415", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "33484.96366468873", "-89254.41632352016", "150000.0", "150000.0", "150000.0", "150000.0", "-96679.00963886667", "150000.0", "150000.0", "-76503.35118586777", "-150000.0", "-150000.0", "-72093.35693269274", "-34095.775430107395", "-32366.937385652152", "-150000.0", "-80253.63293736946", "83794.89849489101", "150000.0", "73261.93508861198", "149424.34857526378", "48466.57219632031", "34836.333112715234", "-7367.688008610124", "-146459.11203704518", "52435.34591953989", "-22273.638512355745", "64422.0018618652", "102941.52776806646", "-119867.95008262043", "-150000.0", "-147598.9542117486", "-150000.0", "-14381.184720082474", "9597.73855739036", "150000.0", "150000.0", "150000.0", "150000.0", "-6331.173002034122", "69888.22278205579", "-149080.63994109188", "1967.1267027323888", "-15799.576160679006", "-104653.04301310377", "-118936.12990787742", "-104255.59906084387", "-150000.0", "4479.529725449835", "-24086.682127970216", "100524.75699172834", "149756.73269620346", "112019.95464258347", "148145.94437716829", "91368.05440721316", "87387.15832557148", "62250.09483860666", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-144415.06112133714", "150000.0", "150000.0", "150000.0", "150000.0", "150000.0", "150000.0", "-101013.15222454915", "-150000.0", "-150000.0", "-150000.0", "-150000.0", "-66407.9516776335", "150000.0", "150000.0", "150000.0", "150000.0", "-31772.866921870405", "-150000.0", "-150000.0", "-125773.00761734291", "150000.0", "150000.0", "-96348.81165343396"], "info": {"R": 10, "tail_bound": 8.113193708253661e-07, "point_upper": 4.633173497486007e-06, "evals": 1281, "gap": 2.4929296153790623e-08}, "root_upper": 0.7897062205875915, "certification": {"R": 10, "tail_bound": 8.113193708253661e-07, "point_upper": 4.633173497486007e-06, "evals": 1281, "gap": 2.4929296153790623e-08}}, "56": {"d": 56, "a": ["1.002", "1.013806841604004", "1.025752806470146", "1.0378395339260897", "1.0500686826161485", "1.0624419307288973", "1.0749609762274686", "1.0876275370825617", "1.1004433515081973", "1.11341017820025", "1.1265297965777912", "1.1398040070272772", "1.1532346311496127", "1.1668235120101274", "1.180572514391496", "1.1944835250496408", "1.2085584529726485", "1.2227992296427368", "1.2372078093013097", "1.2517861692171355", "1.266536309957684", "1.2814602556636625", "1.2965600543267837", "1.311837778070811", "1.3272955234359116", "1.3429354116663623", "1.3587595890016444", "1.37477022697097", "1.3909695226912764", "1.4073596991687347", "1.4239430056038087", "1.44072171769991", "1.4576981379756901", "1.4748745960810103", "1.492253449116637", "1.5098370819577038", "1.5276279075809838", "1.5456283673960185", "1.5638409315801507", "1.5822680994175011", "1.6009123996419432", "1.6197763907841174", "1.6388626615225341", "1.6581738310388148", "1.6777125493771181", "1.6974814978078023", "1.7174833891953716", "1.7377209683707588", "1.7581970125079946", "1.778914331505315", "1.7998757683707605", "1.821084199612316", "1.8425425356326526", "1.8642537211285157", "1.886220735494823", "1.908446593233521", "1.930934344367263", "1.9536870748579573", "1.976707907030251", "2.0"], "c": ["150000.0", "150000.0", "93994.11354864003", "-150000.0", "-150000.0", "-150000.0", "133707.13672728438", "150000.0", "150000.0", "80034.42788439768", "-150000.0", "-150000.0", "-150000.0", "-101775.29803280944", "150000.0", "150000.0", "150000.0", "150000.0", "-119328.49469941272", "-150000.0", "-150000.0", "-150000.0", "46238.927639736205", "150000.0", "40853.84542295953", "150000.0", "150000.0", "-93015.61435764625", "-150000.0", "-150000.0", "-122707.71576365661", "150000.0", "45718.80057698296", "84916.2762386261", "51265.17847240646", "-4775.335957516849", "-47434.3251529698", "-86619.17585526676", "16855.447839732446", "24910.322446956747", "18342.279547004735", "-9062.033682042005", "-464.49661488010076", "-11381.95001757866", "-33743.17324860779", "-10023.307728758202", "-34423.2733951143", "150000.0", "150000.0", "-150000.0", "-150000.0", "-80512.0910607615", "81671.28724058426", "116028.98330383794", "146786.67567177198", "-150000.0", "-150000.0", "131549.20149618722", "3640.491867749323", "-14146.9289344737"], "lp_estimate": 2.324925668797445e-06, "upper_unrounded": 3.043133476098398e-06, "root_upper": 0.7970542374970266, "certification": {"R": 10, "tail_bound": 3.8956120915606534e-07, "point_upper": 2.3699403296062787e-06, "evals": 641, "gap": 6.731931464921195e-07}}, "60": {"d": 60, "a": ["1.002", "1.013806841604004", "1.025752806470146", "1.0378395339260897", "1.0500686826161485", "1.0624419307288973", "1.0749609762274686", "1.0876275370825617", "1.1004433515081973", "1.11341017820025", "1.1265297965777912", "1.1398040070272772", "1.1532346311496127", "1.1668235120101274", "1.180572514391496", "1.1944835250496408", "1.2085584529726485", "1.2227992296427368", "1.2372078093013097", "1.2517861692171355", "1.266536309957684", "1.2814602556636625", "1.2965600543267837", "1.311837778070811", "1.3272955234359116", "1.3429354116663623", "1.3587595890016444", "1.37477022697097", "1.3909695226912764", "1.4073596991687347", "1.4239430056038087", "1.44072171769991", "1.4576981379756901", "1.4748745960810103", "1.492253449116637", "1.5098370819577038", "1.5276279075809838", "1.5456283673960185", "1.5638409315801507", "1.5822680994175011", "1.6009123996419432", "1.6197763907841174", "1.6388626615225341", "1.6581738310388148", "1.6777125493771181", "1.6974814978078023", "1.7174833891953716", "1.7377209683707588", "1.7581970125079946", "1.778914331505315", "1.7998757683707605", "1.821084199612316", "1.8425425356326526", "1.8642537211285157", "1.886220735494823", "1.908446593233521", "1.930934344367263", "1.9536870748579573", "1.976707907030251", "2.0"], "c": ["150000.0", "150000.0", "54428.38944487351", "-150000.0", "-150000.0", "-87768.23628895651", "150000.0", "150000.0", "150000.0", "-40682.67419522039", "-150000.0", "-150000.0", "-150000.0", "68161.14859902533", "150000.0", "150000.0", "150000.0", "-38732.08567054729", "-150000.0", "-150000.0", "-150000.0", "63097.165652427866", "77858.71877666113", "64906.74253642556", "150000.0", "72759.25343886948", "-17178.837983776462", "-150000.0", "-150000.0", "-51322.86060597567", "72774.8093854167", "141407.37120575964", "-3629.8791822625967", "-736.1248793354592", "-3037.3586015124906", "-61793.30095534255", "11017.112890114127", "46942.536273912905", "-3517.944585388418", "-33715.86600722856", "38374.66079220693", "22801.832357693926", "-103660.56326754551", "-150000.0", "113619.15011284452", "150000.0", "150000.0", "-93551.90322780078", "-150000.0", "-31071.698130024317", "-150000.0", "150000.0", "150000.0", "84110.80913485064", "-150000.0", "-150000.0", "67165.65050993457", "150000.0", "-113975.88808794104", "22417.223827227604"], "lp_estimate": 1.128820317867145e-06, "upper_unrounded": 1.5253349858432789e-06, "root_upper": 0.7999375566994174, "certification": {"R": 10, "tail_bound": 2.8200984842981734e-07, "point_upper": 1.1595588170810315e-06, "evals": 895, "gap": 3.657761687622474e-07}}, "64": {"d": 64, "a": ["1.002", "1.013806841604004", "1.025752806470146", "1.0378395339260897", "1.0500686826161485", "1.0624419307288973", "1.0749609762274686", "1.0876275370825617", "1.1004433515081973", "1.11341017820025", "1.1265297965777912", "1.1398040070272772", "1.1532346311496127", "1.1668235120101274", "1.180572514391496", "1.1944835250496408", "1.2085584529726485", "1.2227992296427368", "1.2372078093013097", "1.2517861692171355", "1.266536309957684", "1.2814602556636625", "1.2965600543267837", "1.311837778070811", "1.3272955234359116", "1.3429354116663623", "1.3587595890016444", "1.37477022697097", "1.3909695226912764", "1.4073596991687347", "1.4239430056038087", "1.44072171769991", "1.4576981379756901", "1.4748745960810103", "1.492253449116637", "1.5098370819577038", "1.5276279075809838", "1.5456283673960185", "1.5638409315801507", "1.5822680994175011", "1.6009123996419432", "1.6197763907841174", "1.6388626615225341", "1.6581738310388148", "1.6777125493771181", "1.6974814978078023", "1.7174833891953716", "1.7377209683707588", "1.7581970125079946", "1.778914331505315", "1.7998757683707605", "1.821084199612316", "1.8425425356326526", "1.8642537211285157", "1.886220735494823", "1.908446593233521", "1.930934344367263", "1.9536870748579573", "1.976707907030251", "2.0"], "c": ["150000.0", "150000.0", "16417.189372568053", "-150000.0", "-150000.0", "-6522.382293491663", "150000.0", "150000.0", "113442.76783305516", "-150000.0", "-150000.0", "-150000.0", "-7729.460052944524", "150000.0", "150000.0", "150000.0", "-70453.29475787457", "-134701.098666691", "-150000.0", "-57899.49248707789", "150000.0", "3982.062597324415", "15166.200020202463", "22936.408993785666", "66720.46281595189", "20827.841497154215", "-69121.30655081075", "-90069.83084427944", "-2791.7572083466457", "-2817.137819400475", "60124.71189607179", "59830.1545741754", "-21016.96067947851", "-72529.24191109522", "-8810.012777828228", "6814.081192923582", "32382.039268317338", "62579.71740210113", "5120.158114488639", "20161.100557336038", "-11419.714537283731", "-150000.0", "-150000.0", "96754.7482497808", "150000.0", "150000.0", "56923.41013826799", "-150000.0", "-150000.0", "-150000.0", "97250.78676500119", "150000.0", "131079.08252176625", "-18578.579602322996", "-150000.0", "-142209.08479256884", "122306.44457646649", "150000.0", "-150000.0", "34959.93868287324"], "lp_estimate": 5.576414284994641e-07, "upper_unrounded": 5.922934015936329e-07, "root_upper": 0.7992743739083016, "certification": {"R": 10, "tail_bound": 2.3387473303171348e-07, "point_upper": 5.908119760938052e-07, "evals": 1281, "gap": 1.4814254998276488e-09}}, "68": {"d": 68, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["44380971641391042.685494383142031782918156617725520595748478377797200000000000000000000000000000", "-4576646682241904.0767491894123657374546426604089563291349671529420000000000000000000000000000000", "-22708379462549814.566481251636265942556028657464327878144674715602000000000000000000000000000000", "48419233593736728.628044009270190918888701863912494611341085687575000000000000000000000000000000", "-55694999054564816.423811686608681309015121114365391947526180991418000000000000000000000000000000", "34058558769507877.272396325748606872552913364885462077834915149036000000000000000000000000000000", "4838604893452698.5795367461632988204979774480290527712084631904340000000000000000000000000000000", "-30655319080600626.773279937211850826052378954036919543947485988629000000000000000000000000000000", "20259720825736384.193255563155850902316905004114994840147898854730000000000000000000000000000000", "17202001286988041.869578157244957198327275488446948796129205224330000000000000000000000000000000", "-46181067049855089.482516703564883049929315141759996841706926060302000000000000000000000000000000", "40943309381375129.533726836110640891149629303899397655809154254938000000000000000000000000000000", "-13054057527700939.213764110048099882056582810733828173536345134951600000000000000000000000000000", "-5154116080776704.6981493864757104540256244883767734857198229963750000000000000000000000000000000", "5500073555619954.7357709450215569404661716591104877651892634222780000000000000000000000000000000", "-21800225317324252.627145499434628488297751383262874746276932644953600000000000000000000000000000", "91025703819119413.675028624829337141982838526486330056746438601478000000000000000000000000000000", "-196731087834575794.71800475805401983642627276838556687507032870072480000000000000000000000000000", "263696967728336887.63888955670983275037233304623173900850084313799100000000000000000000000000000", "-221946136173452661.01078689698201229053168228697606626974708569353900000000000000000000000000000", "79671288715824985.106792716117521936362126944540159094003869536780000000000000000000000000000000", "72305217137970527.493873893555324132671897141707865146660367220726700000000000000000000000000000", "-134973135243071863.38640910586326226294794236764627080009372518881060000000000000000000000000000", "84985277772154755.363178108490383021849850916378169332482268943965500000000000000000000000000000", "10618108601899168.513968504505067253304673860906619941089449805116200000000000000000000000000000", "-58995173720156190.246119538396444594005881989646545828558330345445000000000000000000000000000000", "22031358095520988.310563380880388521122686652940086183480082092954000000000000000000000000000000", "58644763417786346.143952815122649017649866260009984425919828978168400000000000000000000000000000", "-106360426327892720.77777281130981868911181382438888594119921482357380000000000000000000000000000", "76425228616023107.340397249244956839107720830815097147856457180394200000000000000000000000000000", "12652505696541572.921249492028245633921131057207956482361007788840200000000000000000000000000000", "-101872660460081271.01067798444003951395450761134802292773384059162200000000000000000000000000000", "139948779539850057.91624396566156709812077304454127796402251350743800000000000000000000000000000", "-115499971526117669.06684510138181628737133200157052082991154417879940000000000000000000000000000", "54344186113988391.965376406932379978282008521101622615661142620385800000000000000000000000000000", "6276614161559610.6228135659698475391637850395372340330209189500596000000000000000000000000000000", "-42109035368914194.584406767798952430797611791545502637934314059089600000000000000000000000000000", "49784145384188913.357734578833169031227407147512256290833904189167000000000000000000000000000000", "-39708311191515507.968744716642332808866848879597800781000438996456200000000000000000000000000000", "24588028881891346.870762382608360097216063146067714538202726106718000000000000000000000000000000", "-12306429295148379.380005081039155009773960820695118490916445317992400000000000000000000000000000", "5027058163090161.6875716477235481571335548780045475870843149356298000000000000000000000000000000", "-1665611307535156.6853496715828247672728261669676997721058538305160000000000000000000000000000000", "439112359261977.66315191870207491412309270363305460064447781397400000000000000000000000000000000", "-88954868302434.396064161639708319661558580560818952644407133243400000000000000000000000000000000", "13034740475973.385599929805370732309692717236013996908657774602000000000000000000000000000000000", "-1231625931594.5128804801296173253177408015189117152958560752358000000000000000000000000000000000", "56398990139.059287040911104052231320037968550855796628126816000000000000000000000000000000000000"], "sampled_error": 8.004256698328696e-08, "upper_unrounded": 8.091181092308048e-08, "root_upper": 0.7865132759020101, "certification": {"R": 40, "tail_bound": 6.830965564724868e-16, "point_upper": 8.001339816072038e-08, "evals": 1281, "gap": 8.984127623601042e-10}, "method": "high_precision_preconditioned_LP"}, "72": {"d": 72, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["61077232829725467.442920281965233289186812882732377815934695766869126000000000000000000000000000", "-19155071271389990.302896912981692108696735011464567106888883599258886000000000000000000000000000", "2898690378739472.4635251085585381451277305710329381043814202548623720000000000000000000000000000", "13394903390567377.750905104061473365557404051558574664266871830420600000000000000000000000000000", "-22414940542078171.416735620355163065029399129425514768282901575775472000000000000000000000000000", "16617985206558414.768292210873232091828047861904172174477180295352258000000000000000000000000000", "2429578287100085.5445538988414865898227384365344395028817271394813400000000000000000000000000000", "-22719309636303550.366859434128583218023947222675843031325157095769838000000000000000000000000000", "31418879809132803.091107558541895579160678997793267811054327942407136000000000000000000000000000", "-26639991927292137.995453857116086424744225097952137907524639271139040000000000000000000000000000", "17819531803327170.704449683183004248426864030542293574889769110616470000000000000000000000000000", "-13302446185790445.144661558125166785475631611322406912167440167168234000000000000000000000000000", "10130361996216753.771442484252347810109353775141713016262974910652756000000000000000000000000000", "367411591986118.53099078072578241571836016444812419445510148888204200000000000000000000000000000", "-17807897179943432.883440009474884570498368510994685258707022710555960000000000000000000000000000", "28761245643888930.938882882055242925408093674507499410220230312523754000000000000000000000000000", "-20682982887530511.886809156109082213857334840203500307884414520997992000000000000000000000000000", "-2543239726154674.4002174769745240929795871076053006701767490872321540000000000000000000000000000", "22895084648244715.562507577258100099152570061188408351026665560718240000000000000000000000000000", "-26073902876783196.321335033241079880387663547065261975076638313271668000000000000000000000000000", "14312572753820206.890774999560147747546687669217968541043194937551118400000000000000000000000000", "-1062179585519674.2564953481569750189846630276475924520119123118748664000000000000000000000000000", "-3757223406153198.6941192169657671621987396673267881022003960779087308000000000000000000000000000", "954530898405982.23900545668830585564742793376341877902364659713192840000000000000000000000000000", "3064941381598096.8953162760336855560714479698924830023864871790472840000000000000000000000000000", "-4359067106936358.2778418320518768210328833538957757759987090427439400000000000000000000000000000", "3372558291042226.6360618641242314302911397015120230949357587908797000000000000000000000000000000", "-1735771765206814.9438662424968613635162481690264052065925938836110640000000000000000000000000000", "34542204604463.445086763312463817238652605145671150828248167780886000000000000000000000000000000", "1391400831563325.1094251308118135169501374288254012470024745831451680000000000000000000000000000", "-1476346334105967.9816278358471119566354420413763022450803786974120400000000000000000000000000000", "-628909709970120.55209090702061436303439190763477129056786773652155800000000000000000000000000000", "4182031294596993.4717948970825689922559713214307663352556314983297240000000000000000000000000000", "-7191017260700875.7655038312621945951166425538348006539211813597034280000000000000000000000000000", "8130817109231658.7039567145709690930099662074255669848724092980241180000000000000000000000000000", "-7032217460676645.9895907270033563412945350220332846211110323511971200000000000000000000000000000", "5013297632182239.8051219120040158142949297146080009946183047653883880000000000000000000000000000", "-3144554353397910.7014048671416579228710456224848327083070483418190600000000000000000000000000000", "1848386050781951.8147419222381332083485214218488494810992740649226520000000000000000000000000000", "-1051741482307398.0087143186181384741792746424709997856689642743682200000000000000000000000000000", "565403916861702.83090928777938185111476289006633435759048549008884000000000000000000000000000000", "-270430949158045.95761278557566269463205285285689842951138000979000000000000000000000000000000000", "108387869113903.18378736690171393747752956131131925426574909872587200000000000000000000000000000", "-34637936122569.419352529800394479075150472082336389946199045859880000000000000000000000000000000", "8404635497706.2382335189249191434841664660514981116335716797807160000000000000000000000000000000", "-1450021376966.4155846401789296472153777680209455460408559593130400000000000000000000000000000000", "158414616285.11294179222461348713275029837636583953103385229620400000000000000000000000000000000", "-8243670159.0956994776720745959364336200288746339099355202837400000000000000000000000000000000000"], "sampled_error": 2.0208621972983455e-08, "upper_unrounded": 4.450159618912775e-08, "root_upper": 0.7904858592776481, "certification": {"R": 40, "tail_bound": 5.2836450294900246e-17, "point_upper": 2.0091896026168268e-08, "evals": 641, "gap": 2.440970016295948e-08}, "method": "high_precision_preconditioned_LP"}, "76": {"d": 76, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["-19045648177674843.582100743419030436373077820985341474230834583941420000000000000000000000000000", "8672095337946821.9693883598158059349832631789283011286127867874908000000000000000000000000000000", "-8400752300520905.4819428245112344344342410167873305269832723518900000000000000000000000000000000", "8371615880863129.1928678053600680088539762865274920709792309474104200000000000000000000000000000", "-7527230693077523.1848097920757847667773264138991770810469480035232000000000000000000000000000000", "6190903672671282.1745103891164072137134468391859401467460671166341000000000000000000000000000000", "-5796295261503283.9696494930320038971438016491131624872603733466956000000000000000000000000000000", "7489151155532605.4662327052890064213154143479375131659923064610071000000000000000000000000000000", "-10470268345570277.464087027202156493464192396292434163590711910733200000000000000000000000000000", "12428188314534704.011087660294790397493685394700490460052406767847120000000000000000000000000000", "-12329839648724029.386243900216643741373521177172939950224954005519180000000000000000000000000000", "12043722296705959.649967218913691061531903857453283902271410462379800000000000000000000000000000", "-13631328815789777.402237894946826053750542184581895419928526806401960000000000000000000000000000", "15172261034508091.970456502680082793038264483347064895520262298478720000000000000000000000000000", "-11807740153974200.070122902808124344774186952236376092310987634806740000000000000000000000000000", "2743650371075055.4400959659369175830372991496577031596796772474126000000000000000000000000000000", "4641799240862280.4489651049966218249965121842823537493645297137570400000000000000000000000000000", "-1309975000485539.3235637451251717929681619588180421441715633198588200000000000000000000000000000", "-12006995324076708.295197787926211200191340902761770911398623267671620000000000000000000000000000", "22777253413661629.975574132130861834633850925223466172295194488869020000000000000000000000000000", "-18728462894177146.976140418108615924677796750234126522464879042243076000000000000000000000000000", "1383982314158840.7501287259003327713562534256895795528682273678387360000000000000000000000000000", "14414920123295345.468049254452669707020064239743493082461956357541596000000000000000000000000000", "-14844372960632304.184172924732278828630594544891791499305652695559020000000000000000000000000000", "58896127431736.219225922921132143577416292250785042585157243685440000000000000000000000000000000", "16851871007287914.279531875494591510691020101391115597236035990024580000000000000000000000000000", "-22307868058017453.429058274481418379122475862292221613187601888564260000000000000000000000000000", "13394488834647138.994652749548809429460716909708503078038156326729240000000000000000000000000000", "1362159856739379.3025194445463916276409511709566878644896003846431000000000000000000000000000000", "-10234275097162736.562293362792309382269721452703766403370725739865840000000000000000000000000000", "7533293496461455.3054778209314533441993316267389056381001679285029200000000000000000000000000000", "3142283298041637.8490159205099501548912473791390443412327055099330000000000000000000000000000000", "-12943050572200162.607533036517583913452836597520315309076632364496800000000000000000000000000000", "14915496148571822.059449691142123410671474946638496774098876097504300000000000000000000000000000", "-8494301478821159.0725527436199796817449589138222679639077418727017000000000000000000000000000000", "-1457931304876367.4626195571219880958178644590376703727952393970351600000000000000000000000000000", "9161996781026966.6833591122680977658174671005057619013398020129451200000000000000000000000000000", "-11787883465156691.871639755120220549773357689967266496209887395492880000000000000000000000000000", "10112027156663741.658998819319925873472253457900556364411610428877140000000000000000000000000000", "-6638698520749764.0043496920668760473025028354478996332221405252424000000000000000000000000000000", "3476076605399268.7530012520757245874982209825078264956104925818347400000000000000000000000000000", "-1467778484546707.0317701398252966077298956025134667315851525585132200000000000000000000000000000", "497423023168221.44709019892342471887210758606305583012435156390470000000000000000000000000000000", "-132913947966580.44966754120993898162955650679056535372931780786530000000000000000000000000000000", "27079871735196.286690416881402182307780408313174110161160970859280000000000000000000000000000000", "-3965273579395.8014185428652228058095049547541274159347964162890000000000000000000000000000000000", "372467586849.40232469381927489091676050712692451044439847030540000000000000000000000000000000000", "-16886920080.656680505738944951076444807145716870328433882407800000000000000000000000000000000000"], "sampled_error": 1.1902947603337423e-08, "upper_unrounded": 4.2705037781961275e-08, "root_upper": 0.799894332842778, "certification": {"R": 40, "tail_bound": 1.6430990160152612e-18, "point_upper": 1.1912071821322381e-08, "evals": 857, "gap": 3.0792965960638894e-08}, "method": "high_precision_preconditioned_LP"}, "80": {"d": 80, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["12295761210547000.697609793549849851515677392404049226590922401703390250000000000000000000000000", "-3953507004248856.4835764250917046386895904244331418411903135043337397000000000000000000000000000", "271913625019822.96380245587742171867640516338072118904740779157895315000000000000000000000000000", "3828486921260544.8621384989102665446996068609167378103495012462672169500000000000000000000000000", "-6392537504552502.8780393400782724718048857490042190180212702915879874200000000000000000000000000", "5268496995769174.0862655604644313572345193303097131939164413918572366000000000000000000000000000", "-835519920558242.43730321215716321309353999777580116108959263162025385000000000000000000000000000", "-3346541317529028.1427807573980823527443509334422060660185058554933425000000000000000000000000000", "3498583808316369.4280476500740993478901019165407804396122472651391813000000000000000000000000000", "338656295876930.38719645217989978524745311043263839865815966524112835000000000000000000000000000", "-4063973141080887.9664980066920319108777188265918654486630474115361097800000000000000000000000000", "4084566171785062.5212844221731550641232571500383112071333159696506524900000000000000000000000000", "-1373595891199368.9904472713789689366175763535073368454424397838449643500000000000000000000000000", "-284156474003831.59570798424505055920548541793936681519182749303751081000000000000000000000000000", "-440602004859721.56055878367554128501313416837445340855459136008729918000000000000000000000000000", "926734620214531.00894743604334259141071871079934646155503026365099440000000000000000000000000000", "1149064065092538.1213813803128675126607033475917337518569444759262545450000000000000000000000000", "-4188908636754965.3667949607037084839704881432802110618588642023440652750000000000000000000000000", "4983231738716327.7146293871956679490544466651942453375405290923941074900000000000000000000000000", "-3273405703563492.3468787289784955482464133218522319418580015055259294550000000000000000000000000", "1667712614053976.8116606140137303234865591179101334186466887279272410830000000000000000000000000", "-1387102465646873.3760936997381968611083629569971672495178983526646102055000000000000000000000000", "686222171954681.41163465117490520869333705313921062843278497302008893470000000000000000000000000", "1652478694997692.1806683061902043535073785294891958527186964092948940300000000000000000000000000", "-3438492359882834.6120333066225481472800359405048997269501911172773470100000000000000000000000000", "1639256464892872.2192610888899461074592970777089656881075605655241033050000000000000000000000000", "3072668856010929.8952649427626959843438734265338508072194834654823540600000000000000000000000000", "-6360677514445641.8103755353162429976530167390789384067842837901665483350000000000000000000000000", "5007275650210094.1961200609262859238875276031031369402121304766620491550000000000000000000000000", "-280968089608596.09787495349553923378873941039632801315108487093598215000000000000000000000000000", "-3937816733656877.5417983433603539267970556324065541269424317859617662150000000000000000000000000", "5296974407509931.9654185319276511099726762263692195735602335468896339500000000000000000000000000", "-4452941882252900.2437916297615213825788813794076251429307066670622944250000000000000000000000000", "3191068640933750.6282999305134137210807026741622272863239268972671845500000000000000000000000000", "-2308183692660626.7497550083260874124417296494990847569675922969082008250000000000000000000000000", "1496631473826443.8802555171768039552887134273431179842932891949755553800000000000000000000000000", "-449151396951546.47660855092722198426691010044962586856386691682209995000000000000000000000000000", "-576615282123639.64258398451337001297844491993018380618584146641445102000000000000000000000000000", "1139086548564342.1539493446979525429428186079692554311592509326600759400000000000000000000000000", "-1124508972633322.2583674824221468806206167905650052882865547005590724250000000000000000000000000", "777572629664343.05822715238789113195530095613345979985008303123350105000000000000000000000000000", "-405594782009786.44078119946933259011045496145409883206559829621541835000000000000000000000000000", "162737978296697.29473394460216493227197561636482162492307502033374040000000000000000000000000000", "-49949937450640.680273677882856081584511051462095502195308913069296800000000000000000000000000000", "11423868001938.868357648744260617460517579855569405109800648947979300000000000000000000000000000", "-1843624938148.4072999828344263305149834891616242142990545499801008700000000000000000000000000000", "188000446959.03119468144788977610567002286084484283545778009323448000000000000000000000000000000", "-9136033778.0392682954930507044793843724561227247747506047334072750000000000000000000000000000000"], "sampled_error": 6.55785920614134e-09, "upper_unrounded": 1.197353752893151e-08, "root_upper": 0.7961186190351404, "certification": {"R": 40, "tail_bound": 8.272738257221032e-20, "point_upper": 6.54620836856766e-09, "evals": 641, "gap": 5.42732916036385e-09}, "method": "high_precision_preconditioned_LP"}, "84": {"d": 84, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["-796791996977355.99456450615027881813085312875536938007814036441010500000000000000000000000000000", "585708960657438.85102536409301378961466917274058002750455784507008600000000000000000000000000000", "-984614944736755.15176661544294718042231710274694202804901890269923700000000000000000000000000000", "1355276213886121.0677088813362186131729517599937413573923594755171940000000000000000000000000000", "-1198022421067830.3692196844084711930594729817538175373114805089391100000000000000000000000000000", "221337812563315.02686548170392859457671790342919252992200211837343800000000000000000000000000000", "1190716250327424.3128575095089832774056368821455670917192073570497020000000000000000000000000000", "-2088090200143372.8747981588937005842698446148552487387199952495672940000000000000000000000000000", "1846629623581434.3693352725663259391794030670194494282270084006249120000000000000000000000000000", "-939874339667570.08289953264097780843788368308997120374277726744486000000000000000000000000000000", "465472444109367.33396040711944857543243299441384377662321413972392300000000000000000000000000000", "-789887543142471.21723874723025291973262787984956618408376352780075400000000000000000000000000000", "1121404367515330.0382085499767824513754114244180437073560938909915290000000000000000000000000000", "-794697745741521.55566867474127548311692680903749393923056713845537500000000000000000000000000000", "493833084485075.02144658055695760651632354654998789804487163884873000000000000000000000000000000", "-1404368673822369.2988625730956187367396577699467179831351196595223820000000000000000000000000000", "3291623374406683.4634499526518582234389334709098310308564872565889060000000000000000000000000000", "-4413891191758000.6909685505832564568913625875811927705425391625578160000000000000000000000000000", "3684628670799540.6348322217196841994604661315122109998091641911983176000000000000000000000000000", "-2060948002419028.2921033709722088412469350401135644236861625088982300000000000000000000000000000", "1141604189331195.6602411308804341278648570581083719273966307778507030000000000000000000000000000", "-1075760274470682.5941186589746683036276826970847886443093488262871480000000000000000000000000000", "807946821274212.81140026180600651773944272814765534527053618471726500000000000000000000000000000", "-38679785698937.888780010229167834667129641631531453364461967997017000000000000000000000000000000", "-225290504265783.67777727643840074737110817427886021223301008407026500000000000000000000000000000", "-699772950178971.53274607557189526367260586918478580055833315515556000000000000000000000000000000", "1937299403197674.2364679134637892161970258356770752633025460151037740000000000000000000000000000", "-2057352562880192.4419804464194217041457761677020507468379453131536480000000000000000000000000000", "835609814417676.00471440618989695404088889131613399279969849337928200000000000000000000000000000", "625234212739726.03657340732331874558287042093021811373269156830595100000000000000000000000000000", "-1332784998255766.5229519386806728259308924271851954697387049092397600000000000000000000000000000", "1365710191619536.9302815614641556298988701563428448649839164038317290000000000000000000000000000", "-1349924380838731.6831594714679739398577694963987151113014773720579420000000000000000000000000000", "1484897800370605.4917255114766611655867236237623321661560178835320570000000000000000000000000000", "-1408324236034049.1311317618620735973214770202748348043434059512275650000000000000000000000000000", "821797099370056.11037228336173582993525660203113490624687107138465700000000000000000000000000000", "75061306100093.660136317545008449306586604981757069714961567014730000000000000000000000000000000", "-798577586201315.67800810688852048564932252306916192005825730117772000000000000000000000000000000", "1045939401935398.0879114110703416648590324132377461477421108609960150000000000000000000000000000", "-877370198342041.00566015742440380772776099916864515092239408828349300000000000000000000000000000", "547264608461901.92248290403680945757067259231984106958814050911601000000000000000000000000000000", "-264738550599804.63343647890675721031503015419308000305880539447150000000000000000000000000000000", "100152150900759.95205196085861530148245846254815087609859293855891000000000000000000000000000000", "-29305698856034.114048237536105752570724953701128281437112136135690000000000000000000000000000000", "6439587056194.8671166809781388835007986660912012114601096702065880000000000000000000000000000000", "-1004104251493.3209726810842612421835708321960923876719991732310100000000000000000000000000000000", "99329887776.304097762203109055637835624262235358486549952837800000000000000000000000000000000000", "-4696111453.4846828854205870474412005703614504624151504380512000000000000000000000000000000000000"], "sampled_error": 1.9068044570623165e-09, "upper_unrounded": 4.139217233442297e-09, "root_upper": 0.7946966959280011, "certification": {"R": 20, "tail_bound": 2.0852378309342424e-09, "point_upper": 1.90695537515245e-09, "evals": 321, "gap": 2.232261858289847e-09}, "method": "high_precision_preconditioned_LP"}, "88": {"d": 88, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["547920518881085.51233873381140209384751942223351718029837186507692096000000000000000000000000000", "-169647089656465.26807338905047867023177533010142711102871528384920040000000000000000000000000000", "-29613510910966.102969391177185685969191034514334121350969870120629400000000000000000000000000000", "243956021743563.29617721264566531468958497915377337689552690430483360000000000000000000000000000", "-322270964019316.99626205690345135865949642889024681525838760466373960000000000000000000000000000", "138657620925413.81552100909198785103002707354974741931065243126080772000000000000000000000000000", "199094303176962.95155748166307903101480826259107292233827653181623760000000000000000000000000000", "-360053706274703.49322120466313308763572068326652690529879517061537500000000000000000000000000000", "124365608430905.33026395342447576306630025488331996779211350468544300000000000000000000000000000", "307575358825347.01431276733656843408814185559305140098978197144246680000000000000000000000000000", "-508305835135511.61162027186395224593756898248967481095077998318394410000000000000000000000000000", "378078982437746.84742079830991375504990181803255872162647338260381738000000000000000000000000000", "-289380526689281.49086743944986528778526426857637629578874987039724220000000000000000000000000000", "509181600399935.42643013896508743016133572017483325049833779008402980000000000000000000000000000", "-714754551908665.46355061147428633805444217994091104363660996577650860000000000000000000000000000", "440972829297361.33405605247474694205230112385039396290503195629659520000000000000000000000000000", "121079101135504.26620406887685828363788389079449309606650946230261772000000000000000000000000000", "-272550644018624.46286157392678237142475476870819452904199422102553120000000000000000000000000000", "-172925270443978.98498642298646151451490200923013436176264869589969680000000000000000000000000000", "541403703106990.83274653795586744613260377976141790756131115353388620000000000000000000000000000", "-252748899048134.58628452073159739456608657850052374841275186789123060000000000000000000000000000", "-324777843786042.42688189562333833986426858285172790167005397540705080000000000000000000000000000", "421615177581580.44996261583583950050395123192135985805316722301769170000000000000000000000000000", "25927828890242.401388649913368594374654543418674127229323141901330160000000000000000000000000000", "-352077739464532.02144312637835136386115834161234638133901529937958084000000000000000000000000000", "141590926764809.62178513990379006112502257104421845707471149755471128000000000000000000000000000", "294529924378718.70327943622155473134950061890224156183511002834374660000000000000000000000000000", "-487718368704101.02218740046961781049883502357392024637783538233653648000000000000000000000000000", "400940056297475.14946399995860195325977547148762867609185801642803408000000000000000000000000000", "-255424448619033.90246451270788183243920274373600363800571977028284080000000000000000000000000000", "109845847321852.35688525734370441813404735864862170629273923530345200000000000000000000000000000", "102507782210091.36239469012088562890858572917589014435722377969066532000000000000000000000000000", "-293665645526011.90021689630486672110328896877816330300553562209787532000000000000000000000000000", "264961591779755.58821040343077626955345323584083400678169171638179420000000000000000000000000000", "-7595654913134.1992071062874066202846846760689938822213394781303888400000000000000000000000000000", "-235902445028968.22615486803679387692451509371720209217097732566156924000000000000000000000000000", "245235441638211.08556034475897110064257530121712363808098728669767884000000000000000000000000000", "-46416611081513.132742125243849150023009252916723015001546537961118960000000000000000000000000000", "-167081707491998.18694381527240494276048161273556605273927442447927412000000000000000000000000000", "252337740754836.07737141008723584297668275956251522489819619063915700000000000000000000000000000", "-211291268754337.20089584044144496385084014307485283494167182044485320000000000000000000000000000", "124046559407536.28492760338942444958703462004874902722699133346141520000000000000000000000000000", "-54069248534352.047457939465793051770829455062046477914156388651120000000000000000000000000000000", "17651578662604.113151130106977399213652710077193876966396234610662720000000000000000000000000000", "-4232784351201.8368964385070548782227852505683424156641177427948614000000000000000000000000000000", "708698299024.47942528751162020093859569114059234914976104803592640000000000000000000000000000000", "-74359889753.427285449128130721175292037206133210911418776772570960000000000000000000000000000000", "3693295459.9337547918250232525657132979548523092613311776434303200000000000000000000000000000000"], "sampled_error": 6.623145572469764e-10, "upper_unrounded": 1.0826948100527382e-09, "root_upper": 0.7908960144047478, "certification": {"R": 20, "tail_bound": 3.802586515708699e-10, "point_upper": 6.618134492146191e-10, "evals": 321, "gap": 4.208813608381191e-10}, "method": "high_precision_preconditioned_LP"}, "92": {"d": 92, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["324026401185435.96655884423000698109267721405997778542993696052174100000000000000000000000000000", "-111884538324793.11406140577338742377721559077138149185928449354230320000000000000000000000000000", "2599073185165.3701944142114004376688439055001152182237597648359480000000000000000000000000000000", "144382922059074.92377567935151638887138910400565700950743214395856900000000000000000000000000000", "-270962126112293.59135384344377518552551963812186377947869603021660700000000000000000000000000000", "299333849783246.07338552853346331735086133699023345183678821301853820000000000000000000000000000", "-227207896910951.56921717720568529507161316382699540113890744659590420000000000000000000000000000", "155498355881074.34602374658362935131055046368233162329638214715717040000000000000000000000000000", "-178604763417567.86937897309662031147313367308265423521044962293810380000000000000000000000000000", "251337883832696.76166524915509421622745844298618030225525025007088400000000000000000000000000000", "-238601467842568.85508314742706577270914123558042621516982037093821460000000000000000000000000000", "125476245113377.27959818094397852664074838883854937483469844468907400000000000000000000000000000", "-77288140763161.066790928662980952975608713239142648277163173089037800000000000000000000000000000", "196955031061734.98347399578009567760557131663675912844883132150489740000000000000000000000000000", "-315860575549284.98927640369843940074584589983802604976446932738723380000000000000000000000000000", "194067084163043.98427469144509850785745110635008663108495940661424460000000000000000000000000000", "113918407832065.44781061156329445192400866781641020625346400905270506400000000000000000000000000", "-307630888928967.40199452960823440713476232254815463628709802285463040000000000000000000000000000", "274535251459804.89916490401445770072183427767064020469304845674150040000000000000000000000000000", "-232040735352443.28117718367661738121165398523099007784001230847792320000000000000000000000000000", "340427736086151.61577116020981549768139337296297811067580177956507580000000000000000000000000000", "-428942817378373.99028190862610632987292227336417358893979436829805128000000000000000000000000000", "305635929826389.75517787307559067174005959632982732430476561468899904000000000000000000000000000", "-129524849075675.01562568843962506642592818078502061204208720908996380000000000000000000000000000", "187156582572151.52839246497920828648372524248400713355226837372277700000000000000000000000000000", "-407961800926269.48378062910961881663965512466573054346545164670270880000000000000000000000000000", "420572962952815.09111518495508139466877288277449694870138502584689080000000000000000000000000000", "-98602037908469.753057986377311627685353809524698444888639409773717800000000000000000000000000000", "-236997168774944.55585760135233641564759187431460735532414637846332620000000000000000000000000000", "257210982649537.72789770062992868118629813135344760739589146096146960000000000000000000000000000", "-46420895337680.969671029958816774583244060247568808652700162368601800000000000000000000000000000", "-65907751317177.616867277235399532726752238947915402640371866235516720000000000000000000000000000", "-45744847363683.174254790364377536561959337145315745321714519544482800000000000000000000000000000", "197364896422323.60192133455939452112184482304712140780131985814938020000000000000000000000000000", "-186382138796667.26651294154377673684823477130845970699065637134210660000000000000000000000000000", "29302840177659.674011533116886444122394382653512760329663506546076800000000000000000000000000000", "112987043622526.62259096953276074071840258582522510588346331751127270000000000000000000000000000", "-138976844335350.37644422682935902266710494632368210597683453241062400000000000000000000000000000", "77735878293569.255527859547045051524141515929195194994046598485615200000000000000000000000000000", "-9334526942562.0763057469515823020357057989260694841679515422946854000000000000000000000000000000", "-22495717127818.087345667899278680022363929562343746211257875301525200000000000000000000000000000", "22774588835255.039461179605116252849582815103485953098384998542081000000000000000000000000000000", "-12612752834487.240591060653596770473082496208134256629701459813455200000000000000000000000000000", "4738939513114.0364139768876217576324928591682393756379570406794120000000000000000000000000000000", "-1248484864005.3419167464942350553094362277548594719161147502216438000000000000000000000000000000", "223756814220.16779667849298439402498341239513940158976107080334600000000000000000000000000000000", "-24727141878.969978278849061909166281229773072539111900270383189200000000000000000000000000000000", "1279558487.3580244698537811003099158606686027641769658537498492000000000000000000000000000000000"], "sampled_error": 2.558794059763551e-10, "upper_unrounded": 3.909494037703198e-10, "root_upper": 0.7902062144836889, "certification": {"R": 20, "tail_bound": 1.1491669500103054e-10, "point_upper": 2.528963019642089e-10, "evals": 321, "gap": 1.3805310180611088e-10}, "method": "high_precision_preconditioned_LP"}, "96": {"d": 96, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["-204205695942241.96567972641002229851812860725544915270707173938093660000000000000000000000000000", "105607065817565.93015955071052171681947794438876806225117973065621300000000000000000000000000000", "-110565042230201.30573709780910626215234670677639645500384791709107160000000000000000000000000000", "105302524945419.15739433163776266550590273662090997871104740893403300000000000000000000000000000", "-56771362938104.526855345121185583342428641348368938258238043962553000000000000000000000000000000", "-46824165861776.880391727750343790962675486173383215968888251058010600000000000000000000000000000", "168402312597383.61619037936430788008555851603466381487906915761218600000000000000000000000000000", "-227611242026549.09447515273174563123922849073239639693268879649164100000000000000000000000000000", "165730366274723.96199308226451762058003813806663719093485652527232330000000000000000000000000000", "-17477051741916.752546915793062641826990288066567781857673629685914500000000000000000000000000000", "-103720339797027.75431500821841068036447869689929939326791069300444144000000000000000000000000000", "114170626017898.24599811184224760056664801755180107840209138137655520000000000000000000000000000", "-39685925483539.386994567561106416712266030076383302005815657818701600000000000000000000000000000", "-33417656375170.714297147105505627617797788424282508801563872330880400000000000000000000000000000", "62252312117857.335000665857741467013391259553592831661350251567458700000000000000000000000000000", "-62412317216835.419173482534562321168577338542703267660536889971378000000000000000000000000000000", "42084489761652.242985791873195686968645164735108589449358357324359800000000000000000000000000000", "12482932816530.124162901183325792281905766446805589556784901881639200000000000000000000000000000", "-81727983286207.735346912169209932926979366316387255473284873842178200000000000000000000000000000", "110105769238024.14801458475684611119796185875288873869694732539011140000000000000000000000000000", "-81252844900362.615666745782948757575871109358185339896233155796092100000000000000000000000000000", "39709019805621.903582061348381149881706194879057830673789693690829200000000000000000000000000000", "-14415896779573.354438234233571110390256311661082385035287802828501860000000000000000000000000000", "-21510816825462.154830493438922457751611808526876708060002554693241970000000000000000000000000000", "84025809157085.070797350369378979217752280718834975429835474791981700000000000000000000000000000", "-128222383245787.17446356297530780242089526262491638314217252436754580000000000000000000000000000", "107633258859102.39892069739100960986704989618655440369127222191586840000000000000000000000000000", "-40393775091455.935425224307912791947016291815512447757301046104724700000000000000000000000000000", "-23972110814104.398485981158898046562137133348065470746491798415934300000000000000000000000000000", "66337865533512.299226859508492196764918982719049697310737796868835400000000000000000000000000000", "-86596133387132.767231177148426022264537328544826222837678040785477200000000000000000000000000000", "65544247979102.305247301637861575717465405483428753140174096976594500000000000000000000000000000", "11334438928002.932189761298635730051815806142970765978279363117016800000000000000000000000000000", "-106327372556969.30135474373971717267327008483956483880578255978242030000000000000000000000000000", "150449910998139.34073911814874242149989102725242737814429677949833330000000000000000000000000000", "-115610848505173.02509292389298794422448025022758849732304670188325030000000000000000000000000000", "40545735742926.092168596065227399053997831951935204991142452785951600000000000000000000000000000", "16793637264294.424465822826428233828527966184146532744254177376380400000000000000000000000000000", "-32762371245925.286341452157276243761248633634933026121532895285985800000000000000000000000000000", "21903591687582.374787658482811882982954514642012102124151942942512200000000000000000000000000000", "-7368921569471.3497247858730480198986932531694559773507903786599384000000000000000000000000000000", "-105964549786.50892149453119870558323378546997273189825803828013160000000000000000000000000000000", "1547652859687.9214258114927329341555079901245320205863297485522676000000000000000000000000000000", "-909011251926.64450784268468817491727160838903163028283446236653260000000000000000000000000000000", "300373718607.65107969178240282185889303474533951652259430302154740000000000000000000000000000000", "-62267544410.670757599452229487527427051085996753899555374383426000000000000000000000000000000000", "7638011136.7518428658831868909055147328297560672494422447402417000000000000000000000000000000000", "-427987531.55997839010445322523834477984830628570089411734634340000000000000000000000000000000000"], "sampled_error": 1.1661985781891343e-10, "upper_unrounded": 1.6363387355520496e-10, "root_upper": 0.7907900191430954, "certification": {"R": 20, "tail_bound": 4.3930178735294227e-11, "point_upper": 1.1662319420046035e-10, "evals": 321, "gap": 4.701067935474461e-11}, "method": "high_precision_preconditioned_LP"}, "100": {"d": 100, "a": ["1.002", "1.0168435844666106", "1.0319070611486076", "1.0471936875197168", "1.062706769309737", "1.0784496612194054", "1.0944257676458504", "1.1106385434187909", "1.1270914945476431", "1.1437881789796953", "1.1607322073695114", "1.1779272438597357", "1.195377006873462", "1.2130852699183425", "1.231055862402608", "1.2492926704631762", "1.2677996378060281", "1.286580766559034", "1.3056401181374129", "1.3249818141220118", "1.3446100371505973", "1.3645290318223513", "1.3847431056157617", "1.4052566298201172", "1.4260740404807954", "1.447199839358557", "1.468638594903052", "1.4903949432407444", "1.512473589177474", "1.5348793072158708", "1.5576169425878383", "1.5806914123023355", "1.6041077062086784", "1.627870888075594", "1.6519860966862592", "1.6764585469495612", "1.7012935310278217", "1.7264964194812245", "1.7520726624291987", "1.7780277907290063", "1.8043674171717876", "1.8310972376963282", "1.8582230326208016", "1.885750667892766", "1.9136860963576696", "1.9420353590461559", "1.9708045864804333", "2.0"], "c": ["-104039045169424.07705058248570568446879488288614337831468938159758963000000000000000000000000000", "42785980118978.623728205549521628883422929661268373765635914942474200000000000000000000000000000", "-16584115754114.502295005822672678577394029173101866221130995081784980000000000000000000000000000", "-25924992218709.681573898183846488055847330186890217362381915738954170000000000000000000000000000", "75230663023124.774454595416367420440946082611112363043463178184986100000000000000000000000000000", "-107139315062597.59534462128985638217512068781080040155068743972945214000000000000000000000000000", "107611999991465.13912475960019802699750775050873385806455657355156780000000000000000000000000000", "-92930134001174.421639549251037775081367259803477150069571583470418840000000000000000000000000000", "90640503713104.234172086456235390082965253655099927940511664967929360000000000000000000000000000", "-90038722185674.468280927704110118686867675180531658875071310604920160000000000000000000000000000", "35757982271836.694617421807072387942336610513648461519897224009921565000000000000000000000000000", "91088172010781.760535484810936566112049099944412481027693676567264040000000000000000000000000000", "-198469086226261.10630279365520953737180151599315087463104504142946660000000000000000000000000000", "160876427598497.93511587828970019421644059254708442523453597962748115000000000000000000000000000", "1204641166354.5419416097479940701036203394750624405089801323879014000000000000000000000000000000", "-100351323822125.11426400794343562851097129242302914395146316204488493000000000000000000000000000", "17562051999305.029506812668549903879807878944936557182512541824857640000000000000000000000000000", "112550926193845.41308775065350008337618417006595372007788849601880434000000000000000000000000000", "-80072033664676.355728635567714545198321758540289478327291659275737470000000000000000000000000000", "-76467699284187.914767904499713862980625050383151864551321594613417800000000000000000000000000000", "102874292075984.82540377865235154885712863041380517352853356170032910000000000000000000000000000", "86631569813553.887057399771279321672906304460175611060286805640874010000000000000000000000000000", "-252082454304553.30405449852143201274329292051983883100456059861614287000000000000000000000000000", "164127951107850.43711210901760394801266107283041490911908338722343184000000000000000000000000000", "60210765890570.473859826757917472886357965466021055400798495724533370000000000000000000000000000", "-123184643389720.65499660132095900209111626098700928685093149403265352000000000000000000000000000", "-45587607209387.372972611721647258308943791568484121758438075505701900000000000000000000000000000", "224518506966119.16144339609518162313199028025258818138593535620541530000000000000000000000000000", "-211238094282688.78681211010623277534075675587802658820054882435873438000000000000000000000000000", "37493837447387.727860301519821593727547769834553524160401521041700870000000000000000000000000000", "137574817489827.82133542219309493677869681244125328458920669108251090000000000000000000000000000", "-213684517970010.87428783424467785684916807157570978520694164175082283000000000000000000000000000", "183501627095585.12993390033119099307984734922197439035035749264518356000000000000000000000000000", "-78838072636663.444758864012666461364075876642870498023505075618917300000000000000000000000000000", "-46376747878589.338826696932687362962756634041327301859866541111526400000000000000000000000000000", "121057522481936.84915285961927859250435919281757445972275047772530213000000000000000000000000000", "-103354146273172.43797601062817105937127223984570865922818936081805145000000000000000000000000000", "20460390005588.789939341226896118368607660405735709512010066066795600000000000000000000000000000", "59359411762612.499050876010920235018667054095465883477432159908061700000000000000000000000000000", "-90211772015401.931892058233841236732395815556480852033592669114788460000000000000000000000000000", "75423256525779.585212691952070953086899945321008884256567873181510520000000000000000000000000000", "-44189909137242.418315799023409371178971568535050963450324768909418600000000000000000000000000000", "19203702778810.605602688818451256497021198218893830016709329211295600000000000000000000000000000", "-6240504745185.3735270025153141234120915264364909585574392126200028700000000000000000000000000000", "1486642160346.2124352804093993030282024425296060361108349581126451700000000000000000000000000000", "-246751391354.69851697201235892250119025666204535361028136728203627000000000000000000000000000000", "25610670939.054430437017283218189325849326484716162831605244413720000000000000000000000000000000", "-1255670999.4842325169299106136667989593054734888917347793820277600000000000000000000000000000000"], "sampled_error": 7.241503033143424e-11, "upper_unrounded": 1.4424477812497535e-10, "root_upper": 0.7972435255417231, "certification": {"R": 20, "tail_bound": 1.1963068157385344e-11, "point_upper": 7.113865624468907e-11, "evals": 321, "gap": 7.310612188028628e-11}}}
''')


In [ ]:
import csv
CSV_PATH = ROOT / "results.csv"
CSV_COLUMNS = ["d", "I_root_upper_bound", "derived_lower_bound",
               "derived_over_sqrt_d_lower_bound", "target_root",
               "target_achieved", "candidate_origin"]

def write_results_csv(rows):
    """Save the certified bounds printed above, preserving six decimal places."""
    temporary = CSV_PATH.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                "d": row["d"],
                "I_root_upper_bound": f"{row['root_upper']:.6f}",
                "derived_lower_bound": f"{row['derived_lower']:.6f}",
                "derived_over_sqrt_d_lower_bound": f"{row['derived_normalized_lower']:.6f}",
                "target_root": repr(row["target_root"]),
                "target_achieved": row["target_achieved"],
                "candidate_origin": row["candidate_origin"],
            })
    temporary.replace(CSV_PATH)

def download_results_csv():
    """Call again manually to download partial results after interrupting a run."""
    if not CSV_PATH.exists():
        print("No results.csv exists yet.", flush=True)
        return
    try:
        from google.colab import files
    except ImportError:
        print(f"CSV saved at {CSV_PATH}; automatic download is available in Colab.", flush=True)
        return
    try:
        files.download(str(CSV_PATH))
    except Exception as exc:
        print(f"CSV is saved at {CSV_PATH}, but the browser download failed: {exc}. "
              "Use Colab's Files panel to download it.", flush=True)

if MODE not in ("auto", "verify_saved", "optimize"):
    raise ValueError("MODE must be auto, verify_saved, or optimize")
if not DIMENSIONS or N_FREQ < 1 or REFINEMENT_STEPS < 1 or EXCHANGE_ROUNDS<1 or EXCHANGE_POINTS<1:
    raise ValueError("Invalid dimension or frequency settings")
if not (A_MIN > 1 and A_MAX > A_MIN):
    raise ValueError("Invalid frequency range or target")
sys.path.insert(0, str(ROOT))
for name in ("kernel", "kernel17", "precond"):
    sys.modules.pop(name, None)
import kernel17 as certificate
import precond as optimizer
from flint import arb, ctx
results = []
write_results_csv(results)
print("Dimensions:", DIMENSIONS, flush=True)
for d in DIMENSIONS:
    if not isinstance(d,int) or d < 2:
        raise ValueError("This implementation requires integer d >= 2")
    target_root = target_root_for_dimension(d)
    if not math.isfinite(target_root) or target_root <= 0:
        raise ValueError("Dimension-dependent target must be positive and finite")
    print(f"\nStarting d={d}; target_root={target_root:.6f}", flush=True)
    try:
        extra_points=[]
        best_certified=None
        previous_proposal=None
        history=[]
        rounds=1 if MODE=="verify_saved" else EXCHANGE_ROUNDS
        for exchange_round in range(rounds):
            print(f"EXCHANGE d={d}: round {exchange_round+1}/{rounds}",flush=True)
            use_saved=(exchange_round==0 and (MODE=="verify_saved" or
                       (MODE=="auto" and str(d) in SAVED_CANDIDATES)))
            if use_saved:
                if str(d) not in SAVED_CANDIDATES:
                    print(f"d={d}: no saved candidate; use MODE='optimize'",flush=True)
                    break
                trial=dict(SAVED_CANDIDATES[str(d)])
                trial["candidate_origin"]="saved verification"
            else:
                ctx.dps=FIT_DIGITS
                try:
                    trial=optimizer.fit(d,N_FREQ,REFINEMENT_STEPS,A_MIN,A_MAX,
                              extra_points=extra_points,initial_candidate=previous_proposal)
                    trial["candidate_origin"]="exchange optimization"
                except Exception as optimization_error:
                    if best_certified is not None:
                        print(f"EXCHANGE d={d}: optimizer failed; keeping the best certified candidate: {optimization_error}",flush=True)
                        break
                    if not ALLOW_SAVED_FALLBACK or str(d) not in SAVED_CANDIDATES:
                        raise
                    print(f"d={d}: optimization failed ({optimization_error}); using saved candidate.",flush=True)
                    trial=dict(SAVED_CANDIDATES[str(d)])
                    trial["candidate_origin"]="saved fallback"
            ctx.dps=CERT_DIGITS
            target=target_root*(1-1e-5)
            try:
                upper,details=certificate.certify(d,trial["a"],trial["c"],CERT_TOL,
                       MAX_CERT_EVALS,target,max_radius=MAX_CERT_RADIUS,
                       time_limit=CERT_TIME_LIMIT,root_tol=CERT_ROOT_GAP,
                       progress_seconds=CERT_PROGRESS_SECONDS)
                root=certificate.up(arb(upper)**(arb(1)/d))
                if not math.isfinite(root):raise ArithmeticError("Nonfinite certificate")
            except Exception as certification_error:
                if best_certified is None:raise
                print(f"EXCHANGE d={d}: certificate failed; keeping earlier certified result: {certification_error}",flush=True)
                break
            trial.update(upper_unrounded=upper,root_upper=root,certification=details,
                         exchange_round=exchange_round+1,target_root=target_root)
            history.append({"round":exchange_round+1,"trial_root_upper":root,
                            "peak_location":details.get("peak_location"),
                            "fit_grid_size":trial.get("fit_grid_size"),
                            "extra_grid_points":list(extra_points)})
            if best_certified is None or upper<best_certified['upper_unrounded']:
                best_certified=dict(trial)
            trial_shown=math.ceil(math.nextafter(root*1e8,math.inf))/1e8
            best_shown=math.ceil(math.nextafter(best_certified['root_upper']*1e8,math.inf))/1e8
            print(f"EXCHANGE d={d}: trial root <= {trial_shown:.8f}; best certified root <= {best_shown:.8f}",flush=True)
            (ROOT/f"round_d{d}_{exchange_round+1}.json").write_text(json.dumps(trial,indent=2))
            checkpoint=dict(best_certified)
            checkpoint['exchange_history']=history
            (ROOT/f"best_certified_d{d}.json").write_text(json.dumps(checkpoint,indent=2))
            if best_certified['root_upper']<target_root:
                print(f"EXCHANGE d={d}: target reached",flush=True)
                break
            if MODE=="verify_saved":break
            if exchange_round+1>=rounds:
                print(f"EXCHANGE d={d}: round limit reached; returning best certified candidate",flush=True)
                break
            peak_points=details.get("peak_points",details.get("exchange_points",[]))
            unresolved=details.get("unresolved_points",[])
            half=max(1,EXCHANGE_POINTS//2)
            incoming=peak_points[:half]+unresolved[:half]+peak_points+unresolved
            new=[]
            for x in incoming:
                x=float(x)
                if math.isfinite(x) and x>=0 and x not in extra_points and x not in new:
                    new.append(x)
                if len(new)>=EXCHANGE_POINTS:break
            if not new:
                print(f"EXCHANGE d={d}: no new certificate points; stopping",flush=True)
                break
            extra_points=sorted(set(extra_points+new))
            previous_proposal=trial
            print(f"EXCHANGE d={d}: adding {len(new)} points; next grid includes r up to {max(extra_points):g}",flush=True)
        if best_certified is None:
            continue
        candidate=dict(best_certified)
        candidate['exchange_history']=history
        upper=candidate['upper_unrounded']
        root=candidate['root_upper']
        details=candidate['certification']
        ctx.dps=CERT_DIGITS
        shown=math.ceil(math.nextafter(root*1e6,math.inf))/1e6
        volume = arb.pi()**(arb(d)/2) / (arb(d)/2+1).gamma()
        derived = (4 * volume * arb(upper))**(-arb(1) / d)
        derived_lower = certificate.down(derived)
        if not math.isfinite(derived_lower):
            raise ArithmeticError("Nonfinite derived bound; increase precision")
        shown_derived = math.floor(math.nextafter(derived_lower * 1e6, -math.inf)) / 1e6
        normalized = derived / arb(d).sqrt()
        normalized_lower = certificate.down(normalized)
        if not math.isfinite(normalized_lower):
            raise ArithmeticError("Nonfinite normalized bound; increase precision")
        shown_normalized = math.floor(math.nextafter(normalized_lower * 1e6, -math.inf)) / 1e6
        met = root < target_root
        candidate.update(upper_unrounded=upper, root_upper=root,
                         certification=details, target_root=target_root, target_achieved=met,
                         derived_lower=derived_lower, derived_lower_rounded=shown_derived,
                         derived_normalized_lower=normalized_lower,
                         derived_normalized_lower_rounded=shown_normalized)
        (ROOT/f"d{d}.json").write_text(json.dumps(candidate, indent=2))
        results.append({"d":d, "root_upper":shown, "derived_lower":shown_derived, "derived_normalized_lower":shown_normalized, "target_root":target_root, "target_achieved":met, "candidate_origin":candidate["candidate_origin"]})
        (ROOT/"summary.json").write_text(json.dumps(results, indent=2))
        write_results_csv(results)
        print(f"d={d}: I^(1/d) (certified upper bound) <= {shown:.6f}; "
              f"(4|B_1|I)^(-1/d) >= {shown_derived:.6f}; "
              f"(4|B_1|I)^(-1/d)/sqrt(d) >= {shown_normalized:.6f}; target_achieved={met}; source={candidate['candidate_origin']}", flush=True)
    except Exception as exc:
        print(f"d={d}: FAILED, no certified value reported: {exc}", flush=True)
print("Output folder:", ROOT)

print("CSV results:", CSV_PATH, flush=True)
if AUTO_DOWNLOAD_CSV:
    download_results_csv()
